# Deep Learning 021 — Improving Network Performance: the Roadmap

Companion notebook to the lesson. "The model isn't good enough" is two completely different
problems wearing one sentence, and **the fix for one makes the other worse**. So the
roadmap starts with a diagnosis, not a technique.

| Symptom | Diagnosis | Levers |
|---|---|---|
| train bad, validation bad | **underfitting** | more capacity, longer training, better activations, better initialisation |
| train good, validation bad | **overfitting** | more data, dropout, L1/L2, early stopping |
| both fine, but slow or unstable | optimisation | normalise inputs, batch norm, better optimiser, tune the learning rate |

Every row below is measured. `numpy`, `pandas` and `scikit-learn`; no TensorFlow needed.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

# few enough training rows that a large network can memorise them, and enough label
# noise that memorising is the wrong thing to do
X, y = make_moons(n_samples=500, noise=0.30, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.7, random_state=0, stratify=y)
sc = StandardScaler().fit(X_tr)
X_tr_s, X_te_s = sc.transform(X_tr), sc.transform(X_te)
print(f"train {X_tr_s.shape}, test {X_te_s.shape}")

def fit(hidden=(64, 64), alpha=1e-8, max_iter=3000, early=False, patience=300, seed=0):
    # alpha near zero and a tight tol so the optimiser is free to memorise - the point
    # of this notebook is to SEE overfitting, not to avoid it
    m = MLPClassifier(hidden_layer_sizes=hidden, alpha=alpha, max_iter=max_iter,
                      random_state=seed, learning_rate_init=0.01,
                      early_stopping=early, n_iter_no_change=patience, tol=1e-9)
    m.fit(X_tr_s, y_tr)
    return m.score(X_tr_s, y_tr), m.score(X_te_s, y_te), m

## Part A — The diagnosis comes before the fix

There is no general "make it better" step. **Which two numbers you are looking at decides
everything.** Sweep capacity and watch the two accuracies come apart.

In [ ]:
print(f"{'hidden layers':>22}{'train':>9}{'test':>9}{'gap':>8}   diagnosis")
for hidden in ((1,), (2,), (4,), (16,), (64, 64), (256, 256, 256)):
    tr, te, _ = fit(hidden)
    gap = tr - te
    dx = ("underfitting" if tr < 0.87 else
          "overfitting" if gap > 0.08 else "healthy")
    print(f"{str(hidden):>22}{tr:>9.3f}{te:>9.3f}{gap:>8.3f}   {dx}")

Read the first column *against* the second, never either alone.

- **Both low** — the model cannot represent the problem. Adding regularisation here makes it
  worse. You need *more* capacity.
- **First high, second much lower** — the model has memorised the training rows, including
  their noise. Adding capacity here makes it worse. You need regularisation or more data.

That is the whole roadmap in one sentence: **the direction of the fix is opposite in the two
cases, so guessing costs you twice.**

Notice also that the largest network is not the worst. Capacity buys the *ability* to
overfit; it does not force it, and the relationship is not monotone.

## Part B — The overfitting levers

Four of them, all run against the same over-capacity model so the comparison means
something.

In [ ]:
base_tr, base_te, _ = fit((256, 256, 256))
print(f"{'lever':<34}{'train':>9}{'test':>9}{'gap':>8}")
print(f"{'nothing (256, 256, 256)':<34}{base_tr:>9.3f}{base_te:>9.3f}{base_tr - base_te:>8.3f}")

for a in (1e-2, 1.0):                                  # L2 - lesson 026
    tr, te, _ = fit((256, 256, 256), alpha=a)
    print(f"{'L2, alpha = ' + str(a):<34}{tr:>9.3f}{te:>9.3f}{tr - te:>8.3f}")

tr, te, m = fit((256, 256, 256), early=True, patience=20)   # lesson 022
print(f"{'early stopping':<34}{tr:>9.3f}{te:>9.3f}{tr - te:>8.3f}"
      f"   stopped at iter {m.n_iter_}")

tr, te, _ = fit((8,))                                  # the blunt version
print(f"{'smaller network (8)':<34}{tr:>9.3f}{te:>9.3f}{tr - te:>8.3f}")

Every one of those works by **making the training accuracy worse on purpose**. That is what
regularisation is: trading fit on the data you have for fit on the data you do not.

There is one lever that does not make that trade.

In [ ]:
# a big, fixed held-out set so the comparison is not dominated by test-set noise
X_big, y_big = make_moons(n_samples=4000, noise=0.30, random_state=99)

print(f"{'training rows':>15}{'train':>9}{'test':>9}{'gap':>8}")
for n in (50, 150, 400, 1200, 3600):
    Xb, yb = make_moons(n_samples=n, noise=0.30, random_state=1)
    s = StandardScaler().fit(Xb)
    m = MLPClassifier(hidden_layer_sizes=(64, 64), alpha=1e-8, max_iter=1500,
                      random_state=0, learning_rate_init=0.01, tol=1e-9,
                      n_iter_no_change=100)
    m.fit(s.transform(Xb), yb)
    tr = m.score(s.transform(Xb), yb)
    te = m.score(s.transform(X_big), y_big)
    print(f"{n:>15}{tr:>9.3f}{te:>9.3f}{tr - te:>8.3f}")

**More data closes the gap from both ends at once** — the training accuracy falls toward the
noise floor *and* the test accuracy climbs. Nothing was given up.

That is why "get more data" heads every list of fixes, and it is also why the rest of the
list exists: you usually cannot get more.

## Part C — The optimisation levers

These do not change what the model *can* represent. They change whether the optimiser
actually gets there. On badly scaled inputs the difference is not subtle — this is lesson
023's dataset, where `Age` spans about 42 and `EstimatedSalary` spans about 135,000.

In [ ]:
df = pd.read_csv("../data/Social_Network_Ads.csv")
Xa = df[["Age", "EstimatedSalary"]].values.astype(float)
ya = df["Purchased"].values
spans = Xa.max(0) - Xa.min(0)
print(f"Age spans {spans[0]:,.0f}, EstimatedSalary spans {spans[1]:,.0f}"
      f"  ->  {spans[1] / spans[0]:,.0f} to 1\n")

Xa_tr, Xa_te, ya_tr, ya_te = train_test_split(Xa, ya, test_size=0.25, random_state=0,
                                              stratify=ya)

def run(A, B, label):
    m = MLPClassifier(hidden_layer_sizes=(8,), max_iter=300, random_state=0,
                      learning_rate_init=0.01)
    m.fit(A, ya_tr)
    print(f"  {label:<16} test accuracy {m.score(B, ya_te):.3f}")

run(Xa_tr, Xa_te, "raw")
s = StandardScaler().fit(Xa_tr)
run(s.transform(Xa_tr), s.transform(Xa_te), "standardised")
print("\nsame model, same seed, same epochs - one line different")

In [ ]:
print(f"{'learning rate':>15}{'train':>9}{'test':>9}{'iters':>8}")
for lr in (1e-5, 1e-3, 1e-2, 1e-1, 1.0):
    m = MLPClassifier(hidden_layer_sizes=(64, 64), max_iter=3000, random_state=0,
                      learning_rate_init=lr, tol=1e-9, n_iter_no_change=200)
    m.fit(X_tr_s, y_tr)
    print(f"{lr:>15}{m.score(X_tr_s, y_tr):>9.3f}{m.score(X_te_s, y_te):>9.3f}{m.n_iter_:>8}")

Both ends fail, exactly as lesson 017 measured: too small never arrives inside the budget,
too large never settles. And notice that the best *test* score here does not come from the
best *train* score — on a noisy problem, an optimiser that converges less completely
sometimes generalises better, which is early stopping arriving by accident.

## The roadmap, now with evidence behind it

1. **Measure two numbers** — training and validation. Never one.
2. **Both bad?** Underfitting. Add capacity, train longer, check the activation (027–028)
   and the initialisation (029–030).
3. **Train good, validation bad?** Overfitting. More data if you can get it; otherwise early
   stopping (022), dropout (024–025), or L1/L2 (026).
4. **Both fine but slow or unstable?** Normalise the inputs (023), batch-normalise the
   hidden layers, pick a better optimiser, tune the learning rate.

## Try it yourself

1. Set `noise=0.05` in Part A and rerun. Does overfitting still appear? What does that say
   about *what* a network memorises?
2. Combine two overfitting levers — L2 *and* early stopping. Do they add, or does the second
   buy nothing once the first is in place?
3. In Part B's data sweep, find the training-set size at which the gap first drops below
   0.05, and compare it to the network's parameter count.
4. Add a pure-noise third feature to Part A. Which lever handles it best, and which does
   nothing?